# Elo Rating Engine

Tujuan:

Notebook ini bertujuan membangun **Elo Rating Engine** yang digunakan untuk menghitung kekuatan relatif setiap tim nasional berdasarkan riwayat pertandingan internasional.
Berbeda dengan *historical feature engineering* yang menghasilkan statistik sederhana seperti *win rate* atau rata-rata gol, Elo Rating merupakan sistem pemeringkatan dinamis yang memperbarui kekuatan setiap tim setelah setiap pertandingan selesai.

Seluruh proses perhitungan dilakukan secara **kronologis**, sehingga rating yang digunakan pada suatu pertandingan hanya berasal dari informasi yang tersedia sebelum pertandingan tersebut dimainkan. Pendekatan ini memastikan bahwa proses pembangunan fitur tetap bebas dari *future data leakage* dan merepresentasikan kondisi prediksi di dunia nyata (*real-world prediction*).
Output utama dari notebook ini adalah histori Elo Rating setiap tim yang selanjutnya akan digunakan pada tahap **Elo Feature Engineering** untuk membangun fitur-fitur prediktif seperti `home_elo`, `away_elo`, dan `elo_difference`.

## 1. Loading

Pertama, kita akan memuat dataset hasil *data preprocessing* yang akan digunakan sebagai dasar perhitungan Elo Rating.
Pada notebook ini hanya beberapa kolom utama yang diperlukan, yaitu informasi tanggal pertandingan, nama kedua tim, serta skor akhir pertandingan.

In [14]:
import pandas as pd
from collections import defaultdict

In [15]:
results = pd.read_csv("../data/interim/results_clean.csv")

In [16]:
# AMBIL YANG DIBUTUHKAN
results = results[
    [
        "date",
        "home_team",
        "away_team",
        "home_score",
        "away_score"
    ]
]

In [17]:
# KONVERSI DATE
results["date"] = pd.to_datetime(results["date"])

In [18]:
# SORT BY DATE
results = results.sort_values("date").reset_index(drop=True)

In [19]:
# LIHAT
results.head()

,date,home_team,away_team,home_score,away_score
0,1872-11-30,Scotland,England,0.0,0.0
1,1873-03-08,England,Scotland,4.0,2.0
2,1874-03-07,Scotland,England,2.0,1.0
3,1875-03-06,England,Scotland,2.0,2.0
4,1876-03-04,Scotland,England,3.0,0.0


Dataset berhasil dimuat dan hanya menyisakan kolom-kolom yang diperlukan untuk membangun Elo Rating. Seluruh pertandingan kemudian diurutkan berdasarkan tanggal guna memastikan proses pembaruan rating mengikuti urutan kronologis. Langkah ini sangat penting karena Elo Rating merupakan sistem yang bergantung pada hasil pertandingan sebelumnya. Dengan demikian, setiap pembaruan rating hanya memanfaatkan informasi yang memang telah tersedia pada saat pertandingan berlangsung.

## 2. Elo Rating Concept

Sebelum membangun Elo Rating Engine, penting untuk memahami konsep dasar dari sistem Elo Rating itu sendiri. Elo Rating merupakan sistem pemeringkatan yang awalnya dikembangkan oleh **Arpad Elo** untuk mengukur kekuatan relatif pemain catur. Seiring waktu, metode ini diadaptasi ke berbagai cabang olahraga, termasuk sepak bola, karena mampu menggambarkan perubahan kekuatan tim secara dinamis berdasarkan hasil pertandingan. Berbeda dengan statistik sederhana seperti *win rate* atau rata-rata gol, Elo Rating tidak hanya mempertimbangkan hasil pertandingan, tetapi juga memperhitungkan kekuatan lawan yang dihadapi. Sebagai contoh, kemenangan atas tim yang memiliki rating tinggi akan memberikan peningkatan rating yang lebih besar dibandingkan kemenangan atas tim dengan rating yang lebih rendah. Sebaliknya, jika sebuah tim dengan rating tinggi kalah dari tim yang jauh lebih lemah, maka penurunan rating yang diterima juga akan lebih besar. Dengan mekanisme tersebut, Elo Rating mampu memberikan representasi kekuatan tim yang terus berkembang mengikuti performa aktual sepanjang waktu.

### 2.1 Elo Rating Workflow
Secara umum, proses pembaruan Elo Rating pada setiap pertandingan mengikuti alur berikut:
1. Mengambil rating kedua tim sebelum pertandingan dimulai.
2. Menghitung probabilitas kemenangan (*expected score*) berdasarkan selisih rating kedua tim.
3. Mengamati hasil pertandingan sebenarnya (*actual score*).
4. Memperbarui rating kedua tim menggunakan rumus Elo.
5. Menyimpan rating terbaru untuk digunakan pada pertandingan berikutnya.

Karena setiap pembaruan selalu menggunakan rating sebelum pertandingan berlangsung, sistem ini secara alami mengikuti urutan kronologis dan tidak memanfaatkan informasi dari masa depan (*future data leakage*).

### 2.2 Elo Rating Formula

Proses pembaruan Elo Rating terdiri dari tiga komponen utama:

- **Expected Score**, yaitu probabilitas kemenangan berdasarkan selisih rating kedua tim.
- **Actual Score**, yaitu hasil pertandingan sebenarnya.
- **Rating Update**, yaitu perhitungan rating baru.

Persamaan Elo dituliskan sebagai berikut:

$$
R_{new} = R_{old} + K \times (S - E)
$$

dengan:

- $R_{old}$ = rating sebelum pertandingan
- $R_{new}$ = rating setelah pertandingan
- $K$ = K-Factor
- $S$ = Actual Score
- $E$ = Expected Score

Nilai **Actual Score** ditentukan sebagai berikut:

| Hasil Pertandingan | Nilai |
|--------------------|------:|
| Menang | 1.0 |
| Seri | 0.5 |
| Kalah | 0.0 |

Semakin besar perbedaan antara hasil yang diperkirakan (*Expected Score*) dan hasil yang sebenarnya (*Actual Score*), semakin besar pula perubahan Elo Rating yang terjadi.

## 3. Elo Configuration

Sebelum membangun Elo Rating Engine, kita perlu menentukan beberapa parameter yang akan digunakan selama proses perhitungan.
Parameter-parameter ini merupakan konfigurasi dasar sistem Elo dan akan memengaruhi bagaimana rating berkembang dari satu pertandingan ke pertandingan berikutnya.
Pada notebook ini digunakan implementasi Elo klasik dengan satu nilai **Initial Rating** dan satu nilai **K-Factor** yang berlaku untuk seluruh tim. Pendekatan ini dipilih agar implementasi tetap sederhana, mudah dipahami, serta dapat dijadikan baseline sebelum melakukan eksperimen yang lebih kompleks pada pengembangan selanjutnya.

In [20]:
INITIAL_RATING = 1500
K_FACTOR = 20

print(f"Initial Rating : {INITIAL_RATING}")
print(f"K-Factor       : {K_FACTOR}")

Initial Rating : 1500
K-Factor       : 20


Konfigurasi Elo Rating berhasil ditentukan sebelum proses pembangunan engine dimulai. Seluruh tim akan memulai kompetisi dengan **rating awal sebesar 1500**, sehingga tidak ada tim yang memperoleh keuntungan atau kerugian sejak awal perhitungan. Selanjutnya, rating akan berubah secara bertahap mengikuti hasil pertandingan yang dimainkan. Nilai **K-Factor sebesar 20** dipilih untuk memberikan keseimbangan antara stabilitas dan kemampuan beradaptasi terhadap perubahan performa tim. Dengan konfigurasi ini, perubahan rating tidak akan terlalu kecil maupun terlalu ekstrem, sehingga lebih sesuai untuk merepresentasikan performa tim nasional dalam jangka panjang.

## 4. Elo Data Structure
Sebelum menghitung Elo Rating untuk setiap pertandingan, kita perlu menyiapkan struktur data yang akan digunakan selama proses iterasi.

Karena Elo Rating bersifat dinamis, setiap tim harus memiliki nilai rating yang terus diperbarui setelah pertandingan selesai. Selain itu, kita juga perlu menyimpan histori perubahan rating agar dapat dianalisis kembali maupun digunakan pada tahap *feature engineering* berikutnya.

Oleh karena itu, notebook ini menggunakan dua struktur utama:

- **`elo_ratings`** untuk menyimpan rating terkini setiap tim.
- **`elo_history`** untuk menyimpan histori perubahan rating sepanjang dataset.

#### 4.1 Current Elo Ratings

In [21]:
from collections import defaultdict

elo_ratings = defaultdict(lambda: INITIAL_RATING)

`defaultdict` digunakan agar setiap tim yang baru pertama kali muncul secara otomatis memperoleh **Initial Rating** tanpa perlu dilakukan pengecekan apakah tim tersebut sudah pernah tersimpan sebelumnya.

Dengan pendekatan ini, proses iterasi menjadi lebih sederhana karena seluruh tim akan langsung memiliki rating awal ketika pertama kali diakses.

### 4.2 Elo History

In [22]:
elo_history = []

Berbeda dengan `elo_ratings` yang hanya menyimpan kondisi terbaru setiap tim, `elo_history` digunakan untuk menyimpan histori perubahan rating pada setiap pertandingan. Setiap elemen pada list akan merepresentasikan satu pertandingan beserta informasi Elo sebelum dan sesudah pertandingan berlangsung.

Setelah seluruh pertandingan selesai diproses, list ini akan dikonversi menjadi `DataFrame` sehingga mudah divalidasi, dianalisis, maupun disimpan sebagai dataset baru.

### 4.3 Melihat Struktur Awal

In [23]:
print(elo_ratings["France"])
print(elo_ratings["Spain"])
print(elo_history)

1500
1500
[]


Struktur data utama untuk membangun Elo Rating Engine telah berhasil disiapkan.

Dictionary `elo_ratings` akan menyimpan rating terkini setiap tim selama proses iterasi berlangsung, sedangkan `elo_history` akan merekam seluruh perubahan rating pada setiap pertandingan.

Pemisahan kedua struktur ini membuat proses perhitungan menjadi lebih efisien karena engine hanya perlu memperbarui satu nilai rating aktif untuk setiap tim, sementara seluruh histori perubahan tetap terdokumentasi secara lengkap.

## 5. Helper Functions
Pada bagian ini, seluruh proses perhitungan Elo Rating dipisahkan ke dalam beberapa *helper function*.

Pendekatan ini dipilih agar setiap fungsi hanya memiliki satu tanggung jawab (*single responsibility*), sehingga kode menjadi lebih modular, mudah dipahami, serta lebih mudah diuji maupun dikembangkan pada eksperimen berikutnya.

Secara umum, proses perhitungan Elo terdiri dari tiga tahapan utama:

1. Menghitung probabilitas kemenangan (*Expected Score*).
2. Menentukan hasil pertandingan (*Actual Score*).
3. Menghitung Elo Rating baru berdasarkan kedua nilai tersebut.

### 5.1 Expected Score

Expected Score merupakan probabilitas kemenangan suatu tim berdasarkan selisih Elo Rating kedua tim sebelum pertandingan berlangsung.

Semakin tinggi rating suatu tim dibanding lawannya, semakin besar peluang kemenangan yang diperkirakan oleh sistem Elo.

Sebaliknya, jika kedua tim memiliki rating yang sama, maka probabilitas kemenangan masing-masing tim adalah **50%**.

Persamaan Expected Score dituliskan sebagai berikut:

$$
E_A = \frac{1}{1 + 10^{(R_B-R_A)/400}}
$$

dengan:

- $R_A$ = rating tim A
- $R_B$ = rating tim B
- $E_A$ = probabilitas kemenangan tim A

In [24]:
def expected_score(rating_a, rating_b):
    """
    Menghitung expected score berdasarkan selisih Elo Rating.
    """

    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

In [30]:
# VALIDASI 1
expected_score(1500,1500)

0.5

In [31]:
# VALIDASI 2
expected_score(1700,1500)

0.7597469266479578

Fungsi `expected_score()` berhasil dibangun dan mampu menghitung probabilitas kemenangan berdasarkan selisih Elo Rating.

Validasi sederhana menunjukkan bahwa dua tim dengan rating yang sama memiliki probabilitas kemenangan sebesar **50%**, sedangkan tim dengan rating lebih tinggi memperoleh probabilitas kemenangan yang lebih besar.

Hasil tersebut sesuai dengan konsep dasar sistem Elo.

### 5.2 Actual Score

Berbeda dengan *Expected Score* yang merupakan probabilitas kemenangan sebelum pertandingan dimulai, *Actual Score* merepresentasikan hasil pertandingan yang benar-benar terjadi.

Dalam sistem Elo, hasil pertandingan tidak menggunakan jumlah gol, melainkan hanya hasil akhirnya. Oleh karena itu, setiap pertandingan dikonversi menjadi nilai numerik sebagai berikut:

| Hasil Pertandingan | Home Team | Away Team |
|--------------------|----------:|----------:|
| Home Win | 1.0 | 0.0 |
| Draw | 0.5 | 0.5 |
| Away Win | 0.0 | 1.0 |

Representasi ini akan digunakan pada proses pembaruan Elo Rating untuk kedua tim.

In [33]:
def actual_score(home_score, away_score):
    """
    Mengubah hasil pertandingan menjadi Actual Score
    untuk tim kandang dan tim tandang.
    """

    if home_score > away_score:
        return 1.0, 0.0

    elif home_score < away_score:
        return 0.0, 1.0

    else:
        return 0.5, 0.5

In [37]:
# VALIDASI 1 : HOME WIN
actual_score(2,1)

(1.0, 0.0)

In [38]:
# VALIDASI 2 : AWAY WIN
actual_score(0,3)

(0.0, 1.0)

In [39]:
# VALIDASI 3 : DRAW
actual_score(1,1)

(0.5, 0.5)

Fungsi `actual_score()` berhasil mengubah hasil pertandingan menjadi representasi numerik sesuai dengan aturan sistem Elo.

Validasi menunjukkan bahwa seluruh kemungkinan hasil pertandingan (*Home Win*, *Draw*, dan *Away Win*) telah menghasilkan nilai *Actual Score* yang sesuai.

Nilai inilah yang nantinya akan dibandingkan dengan *Expected Score* untuk menentukan besar perubahan Elo Rating pada setiap pertandingan.